In [ ]:
import pandas as pd

data = pd.read_csv('matches.csv')

remove_duplicate_data = data.drop_duplicates().reset_index(drop=True)
remove_duplicate_data["rank_a"] = remove_duplicate_data["rank_a"].astype("Int64")
print("\n\n\nНовая таблица без дубликатов матчей: ")
print(remove_duplicate_data.to_string())

data["rank_a"] = data["rank_a"].astype("Int64")

# remove_duplicate_data.to_csv('matches_after_proc.csv', index=False)

# clean_df = data.loc[~(data["rank_a"].isna())].drop_duplicates(subset="match_id").reset_index(drop=True)
# print("\n\n\nПолностью чистая от багов таблица")
# print(clean_df.to_string())

matches = remove_duplicate_data
date_of_cut = "2026-10-06"

history = matches.loc[matches["date"] < date_of_cut]
predict_date = matches.loc[matches["date"] >= date_of_cut]

after_data = history.dropna(subset=["rank_a", "rank_b"])
after_data = after_data.loc[after_data["rank_a"] != after_data["rank_b"]]

n = after_data.shape[0]
k = ((after_data["rank_a"] > after_data["rank_b"]) & (after_data["sets_a"] < after_data["sets_b"])).sum()
k += ((after_data["rank_a"] < after_data["rank_b"]) & (after_data["sets_a"] > after_data["sets_b"])).sum()
p_fav = k / n

print("\nДля дат до 2026-10-06:")
print(f"n = {n}, k = {k}, p_fav = {p_fav}")

print("\n\nmatch_id - ид матча, p_a - шанс победы Игрока_А, fact - победил ли Игрок_А: \n")
for match in predict_date.itertuples(index=False):
    if (pd.isna(match.rank_a) or pd.isna(match.rank_b) or match.rank_a == match.rank_b):
        print(f"--SKIP--")
        continue

    p_a = p_fav if match.rank_a < match.rank_b else 1 - p_fav
    fact = 1 if match.sets_a > match.sets_b else 0
    print(f"match_id={match.match_id}, p_a={p_a}, fact={fact}")


<div style="border-left: 2px solid #f65c5c; padding: 8px 12px; max-width: 120ch;">
    Мне кажется, что доля побед фаворита по группам будет более объективной оценкой. Причем, чем больше будет различаться rank двух игроков, я тем больше будет точность вероятности победы фаворита. Объясняется это тем, что существенная разница в rank соперников отображает разницу уровня игроков. От уровня игроков напрямую зависит результат: чем выше уровень мастерства игрока, тем больше шансов в действительности победить более слабого соперника.Больше всего "неточностей", т.е. когда вероятность победы фаворита высока, а побеждает андердог, будет происходить, когда rank соперников несущественно различается. Нужно только грамотно определить порог этой "неточности", когда мы можем доверять вероятности победы фаворита
</div>

<div style="border-left: 2px solid #88f65c; padding: 8px 12px; max-width: 120ch;">
Пока что короче остановлюсь на gap=16.
В матчах с разрывом в рейтинге 16 мест и больше фаворит побеждает чаще, чем в матчах с разрывом 1–15 мест.
</div>

In [13]:
import pandas as pd
import numpy as np

data = pd.read_csv("matches_after_proc.csv")
data = data[~data["rank_a"].isna()].drop_duplicates().reset_index(drop=True)
data["rank_a"] = data["rank_a"].astype("Int64")

data["gap"] = (data["rank_a"] - data["rank_b"]).abs()
data["gap_group"] = np.where(data["gap"] < 16, "closest", "farthest")

print(data.to_string())

    match_id        date surface player_a player_b  rank_a  rank_b  sets_a  sets_b  gap gap_group
0          1  2026-10-01    Hard    Orlov    Costa      18      42       2       0   24  farthest
1          2  2026-10-02    Clay     Vega      Kim      55      24       2       1   31  farthest
2          3  2026-10-03    Hard      Kim    Orlov      24      18       0       2    6   closest
3          4  2026-10-04    Clay    Costa     Vega      42      55       1       2   13   closest
4          6  2026-10-06    Hard    Costa      Kim      42      24       1       2   18  farthest
5          7  2026-10-07    Clay    Orlov     Vega      18      55       1       2   37  farthest
6          8  2026-10-08    Hard      Kim    Costa      24      42       2       1   18  farthest
7          9  2026-10-09    Hard      Kim    Orlov      24      18       2       1    6   closest
8         10  2026-10-10    Clay     Vega    Costa      55      42       0       2   13   closest
9         11  2026-1

In [ ]:
"""
    Предикт победы фаворита без учета его gap/gap_group.
    В будущем сравню его с предиктом, который учитывает gap/gap_group
"""

date_for_cut = "2026-10-17"

history = data.loc[data["date"] < date_for_cut]
predict_date = data.loc[data["date"] >= date_for_cut]

n_without_gap = history.shape[0]
k_without_gap = ((history["rank_a"] > history["rank_b"]) & (history["sets_a"] < history["sets_b"])).sum()
k_without_gap += ((history["rank_a"] < history["rank_b"]) & (history["sets_a"] > history["sets_b"])).sum()

fav_p_without_gap = k_without_gap / n_without_gap
print(f"n_without_gap={n_without_gap}, k_without_gap={k_without_gap}, fav_p_without_gap={round(fav_p_without_gap, 2)}")

brier_scope_without_gap = 0
for match in predict_date.itertuples():
    if (pd.isna(match.rank_a) or pd.isna(match.rank_b) or match.rank_a == match.rank_b):
        print("--SKIP--")
        continue

    p_A_without_gap = fav_p_without_gap if match.rank_a < match.rank_b else round(1 - fav_p_without_gap, 2)
    fact_without_gap = 1 if match.sets_a > match.sets_b else 0

    brier_scope_without_gap += (p_A_without_gap - fact_without_gap) ** 2
    print(f"match_id={match.match_id}, p_A={round(p_A_without_gap, 2)}, fact={fact_without_gap}")

brier_scope_without_gap /= predict_date.shape[0]
print(f"\nСреднеквадратичная ошибка предикта без учета gap: {round(brier_scope_without_gap, 4)}")


n_without_gap=15, k_without_gap=9, fav_p_without_gap=0.6
match_id=17, p_A=0.6, fact=1
match_id=18, p_A=0.4, fact=0
match_id=19, p_A=0.6, fact=0
match_id=20, p_A=0.6, fact=1

Среднеквадратичная ошибка предикта без учета gap: 0.21


In [ ]:
"""
    Предикт победы фаворита с учетом его gap/gap_group
    В будущем сравню его с предиктом, который не учитывает gap/gap_group
"""

date_for_cut = "2026-10-17"

history = data.loc[data["date"] < date_for_cut]
predict_date = data.loc[data["date"] >= date_for_cut]

n_closest_group = 0
n_farthest_group = 0

k_closest_group = 0
k_farthest_group = 0

for match in history.itertuples():
    if (pd.isna(match.rank_a) or pd.isna(match.rank_b) or match.rank_a == match.rank_b):
        print("--SKIP--")
        continue

    if (match.gap_group == 'closest'):
        n_closest_group += 1
        k_closest_group += (match.rank_a < match.rank_b) & (match.sets_a > match.sets_b) | \
                           (match.rank_b < match.rank_a) & (match.sets_b > match.sets_a)
    else:    
        n_farthest_group += 1
        k_farthest_group += (match.rank_a < match.rank_b) & (match.sets_a > match.sets_b) | \
                            (match.rank_b < match.rank_a) & (match.sets_b > match.sets_a)

p_fav_closest_group = k_closest_group / n_closest_group
p_fav_farthest_group = k_farthest_group / n_farthest_group

print(f"n_closest_group={n_closest_group}, k_closest_group={k_closest_group}, p_fav_closest_group={round(p_fav_closest_group, 2)}")
print(f"n_farthest_group={n_farthest_group}, k_farthest_group={k_farthest_group}, p_fav_farthest_group={round(p_fav_farthest_group, 2)}")

brier_scope_with_gap = 0
print("\n\nПредикт на 4 матча с учетом gap/gap_group:")
for match in predict_date.itertuples():
    if (pd.isna(match.rank_a) or pd.isna(match.rank_b) or match.rank_a == match.rank_b):
            print("--SKIP--")
            continue
    
    if (match.gap_group == 'closest'):
        p_A = p_fav_closest_group if match.rank_a < match.rank_b else 1 - p_fav_closest_group
    else:     
        p_A = p_fav_farthest_group if match.rank_a < match.rank_b else 1 - p_fav_farthest_group

    fact = 1 if match.sets_a > match.sets_b else 0
    brier_scope_with_gap += (p_A - fact) ** 2
    print(f"match_id={match.match_id}, group={match.gap_group}, p_A={round(p_A, 2)}, fact={fact}")

brier_scope_with_gap /= predict_date.shape[0]
print(f"\nСреднеквадратичная ошибка предикта с учета gap: {round(brier_scope_with_gap, 4)}")

n_closest_group=6, k_closest_group=2, p_fav_closest_group=0.33
n_farthest_group=9, k_farthest_group=7, p_fav_farthest_group=0.78


Предикт на 4 матча с учетом gap/gap_group:
match_id=17, group=closest, p_A=0.33, fact=1
match_id=18, group=farthest, p_A=0.22, fact=0
match_id=19, group=closest, p_A=0.33, fact=0
match_id=20, group=farthest, p_A=0.78, fact=1

Среднеквадратичная ошибка предикта с учета gap: 0.1636


<div style="border-left: 2px solid #88f65c; padding: 8px 12px; max-width: 120ch;">
Понял, что пора переходить на реальные данные
</div>

In [190]:
"""
    Из открытых источников нашел данные реальных матчей за 2022 и 2023 года.
    Подчищаю и вывожу только нужные столбцы
"""

import pandas as pd


data2022 = pd.read_csv("atp_matches_2022.csv")
data2023 = pd.read_csv("atp_matches_2023.csv")
data2022["season"] = 2022
data2023["season"] = 2023

data = pd.concat([data2022, data2023], ignore_index=True)
counts = data.groupby("season").size()

rows_before_drop_duplicates = len(data)
data = data.drop_duplicates().reset_index(drop=False)
rows_after_drop_duplicates = len(data)

columns = ["tourney_id", "match_num", "score", "winner_rank", "loser_rank", "season"]
data_before_proc = data
data = data.loc[:, columns]

duplicated_rows = data.duplicated(subset=["tourney_id", "match_num"], keep=False)
if duplicated_rows.any():
    print(data.loc[duplicated_rows].to_string())

print(data.head(10).to_string(index=False))

tourney_id  match_num             score  winner_rank  loser_rank  season
 2022-8888        300        7-6(3) 6-3         11.0        19.0    2022
 2022-8888        299           6-4 6-3         14.0        20.0    2022
 2022-8888        298 7-6(6) 2-6 7-6(5)         19.0         9.0    2022
 2022-8888        297           6-2 6-1         20.0       860.0    2022
 2022-8888        296           6-4 6-0          2.0        11.0    2022
 2022-8888        295       6-4 5-7 6-4         14.0       167.0    2022
 2022-8888        294           6-3 7-5          8.0        33.0    2022
 2022-8888        293           6-2 7-5         42.0       345.0    2022
 2022-8888        292   4-6 6-4 3-0 RET         17.0        33.0    2022
 2022-8888        291    6-4 3-6 7-6(5)         42.0       139.0    2022


In [191]:
data.loc[:, ["tourney_id", "match_num", "score", "winner_rank", "loser_rank"]].to_csv("atp_matches_2022_2023_drop_dup.csv", index=False)

In [ ]:
"""
    Убираю все пустые/некорректные поля + когда матч закончился из-за (RET)/(W/O)/(DIF)
"""

print(data_before_proc.shape[0])
data_before_proc = data_before_proc.loc[~((data_before_proc["winner_rank"] <= 0) | 
                                          (data_before_proc["winner_rank"].isna()) | 
                                          (data_before_proc["loser_rank"] <= 0) | 
                                          (data_before_proc["loser_rank"].isna()))]

print(data_before_proc.shape[0])

data = data_before_proc
data = data.loc[:, columns]
empties = data["score"].isna()
has_letters = data["score"].str.contains(r"[A-Za-z]")

data = data.loc[~empties].loc[~has_letters]
print(f"После фильтра по счёту: {len(data)}")

equal_ranks = data["winner_rank"].eq(data["loser_rank"])
print(f"Матчей с равными рейтингами: {equal_ranks.sum()}")

data = data.loc[~equal_ranks].copy()

print(f"Осталось матчей: {len(data)}")
print(empties.sum() + has_letters.sum())
print(data.shape[0])


5903
5833
После фильтра по счёту: 5648
Матчей с равными рейтингами: 0
Осталось матчей: 5648
185
5648


In [ ]:
"""
    Вывод вероятности победы фаворита по двум группам.
    В конец добавил разницу, чтобы было нагляднее
"""

data["gap"] = (data["winner_rank"] - data["loser_rank"]).abs().astype("Int64")
data["gap_group"] = np.where(data["gap"] < 16, "closest", "farthest")
data["fav_won"] = np.where(data["winner_rank"] < data["loser_rank"], 1, 0)

n = data.groupby("gap_group").size()
k = data.groupby("gap_group")["fav_won"].sum()
p = k / n

summary = pd.DataFrame({"n": n, "k": k, "p": p})
print(summary.round(4))

diff = 100 * (p.loc["closest"] - p.loc["farthest"])
print(f"Разница: {diff:.2f} п. п.")

              n     k       p
gap_group                    
closest    1168   659  0.5642
farthest   4480  2974  0.6638
Разница: -9.96 п. п.


<div style="border-left: 2px solid #88f65c; padding: 8px 12px; max-width: 120ch;">
В сезонах 2022–2023 фаворит по рейтингу выигрывал на 9,96 процентов чаще при разрыве от 16 мест. На сезоне 2024 буду проверять, улучшает ли учёт gap прогноз по MSE(Brier score)
</dic>